# Norway GIP quad-map backtest

Walk-forward backtest of the Hedgeye-style growth/inflation quad model:
*how often would it have called the correct quad 1-4 quarters ahead, using
only information available at the time?*

**How to run:** `Runtime -> Run all`. That's it.

- The notebook clones the repo, installs dependencies, pulls **real data**
  (SSB mainland GDP + CPI, Norges Bank I-44 and USDNOK, Brent from FRED),
  runs the backtest and renders the results below.
- The repo is private, so the clone step will ask for a GitHub **personal
  access token** the first time (github.com -> Settings -> Developer
  settings -> Fine-grained tokens; read access to this repo is enough).
- If any live fetch fails, the notebook falls back to the synthetic demo
  bundle so you still get a full end-to-end run (clearly labelled).
- GDP revisions: without a Norges Bank real-time vintage file the harness
  runs in the flagged **revision-noise** mode (simulated first releases).
  Upload `data/gdp_vintages.csv` (format in `backtest/fetch_data.py`) to
  switch to true vintages automatically.


In [ ]:
# ---- Parameters: edit and re-run ------------------------------------------
START = "2012"       # first as-of year (or date like "2013-06-30")
END = "2025"         # last as-of year
HORIZON = 4          # predict 1..HORIZON quarters ahead
FREQ = "M"           # as-of dates: "M" month-end (slower, more points) or "Q"
REVISION_SIGMA = 0.25  # noise mode: stdev of first-release QoQ revision (pp)
USE_DEMO = False     # True = skip live data, run on the synthetic bundle


In [ ]:
# ---- Setup: clone the repo and install dependencies ------------------------
import os, subprocess
from pathlib import Path

REPO = "SanderHeisan/Inflation-GDP"
BRANCH = "claude/norway-gip-backtest-harness-nln7uw"


def clone(url, token=None):
    # Token goes through a temporary askpass helper, NEVER into the URL or
    # the process argument list, so it cannot leak via error messages.
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:
        helper = Path("/tmp/git_askpass.sh")
        helper.write_text("#!/bin/sh\ncase \"$1\" in\n"
                          "  Username*) echo x-access-token ;;\n"
                          "  Password*) echo \"$GIT_TOKEN\" ;;\nesac\n")
        helper.chmod(0o700)
        env.update(GIT_ASKPASS=str(helper), GIT_TOKEN=token)
    r = subprocess.run(["git", "clone", "-b", BRANCH, url],
                       capture_output=True, text=True, env=env)
    if token:
        helper.unlink()
    return r


if not Path("backtest.py").exists():          # fresh Colab runtime
    if not Path("Inflation-GDP").exists():
        url = f"https://github.com/{REPO}.git"
        r = clone(url)
        if r.returncode != 0:                 # private repo -> need a token
            print("Anonymous clone failed (private repo) - a fine-grained")
            print("personal access token with Contents: Read-only on this")
            print(f"repo is needed (github.com -> Settings -> Developer")
            print("settings -> Fine-grained tokens).")
            from getpass import getpass
            token = getpass("Paste the token: ").strip()
            r = clone(url, token)
            if r.returncode != 0:
                print("\ngit says:\n" + r.stderr.strip())
                print("\nCheck: (1) token copied completely, (2) the token's")
                print(f"'Repository access' explicitly includes {REPO},")
                print("(3) 'Contents' permission is Read-only or higher.")
                raise SystemExit("clone failed")
    os.chdir("Inflation-GDP")

print("working dir:", Path.cwd())
%pip install -q -r requirements.txt
print("setup done")


In [ ]:
# ---- Data: fetch real series (falls back to the synthetic demo) ------------
if not USE_DEMO:
    try:
        from backtest import fetch_data
        fetch_data.main(["--data-dir", "data"])
        print("\nlive data ready in data/")
    except Exception as e:
        print(f"\nLIVE FETCH FAILED ({type(e).__name__}: {e})")
        print("Falling back to the synthetic demo bundle. Results below are")
        print("a harness validation, NOT Norway. Variable codes are resolved")
        print("from SSB table metadata at runtime, so a failure here is")
        print("either a temporary API outage (re-run this cell) or a table")
        print("restructuring. To inspect the table's current codes, run:")
        print("  from quadmap.data_sources import get_table_metadata")
        print("  [(v['code'], list(zip(v['values'], v['valueTexts']))[:8])")
        print("   for v in get_table_metadata('09190')['variables']]")
        USE_DEMO = True
else:
    print("USE_DEMO=True - skipping live data")


In [ ]:
# ---- Run the walk-forward backtest -----------------------------------------
import subprocess, sys

cmd = [sys.executable, "backtest.py", "--start", str(START), "--end", str(END),
       "--horizon", str(HORIZON), "--freq", FREQ,
       "--sigma", str(REVISION_SIGMA), "--outdir", "results"]
if USE_DEMO:
    cmd.append("--demo")

print(" ".join(cmd), "\n")
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("backtest failed - see stderr above")


In [ ]:
# ---- THE HEADLINE: quad probabilities for the year ahead --------------------
# For each of the next four quarters (and the twelve months they contain):
# the probability of each quad. Probabilities are calibrated on the backtest
# itself - "when the model made this call at this horizon, how often did
# each quad actually happen" - so they carry the model's real-world error
# rate, not just its self-confidence. The outlined cell is the point call.
from pathlib import Path
from IPython.display import Image, display

for png in ["quad_probabilities_quarterly.png",
            "quad_probabilities_monthly.png"]:
    p = Path("results") / png
    if p.exists():
        display(Image(filename=str(p)))
    else:
        print(f"{png} missing - check the backtest cell output above")


In [ ]:
# ---- LIVE forecast: refresh with today's market prices ---------------------
# Re-run this cell any day (no need to redo the backtest above): it pulls
# LIVE spots - Brent (FRED), USDNOK + I-44 (Norges Bank, daily), Nord Pool
# area power prices (hvakosterstrommen.no) - pushes them through the CPI
# blocks, and recomputes the probabilities. If oil or power jumped since the
# last CPI print, P(quad 2/3) rises today, not next month. Each run appends
# to results/forecast_history.csv so you can watch the calls drift.
import subprocess, sys

cmd = [sys.executable, "forecast.py", "--outdir", "results"]
if USE_DEMO:
    cmd += ["--demo", "--no-live"]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
else:
    from IPython.display import Image, display
    display(Image("results/quad_probabilities_quarterly.png"))
    display(Image("results/quad_probabilities_monthly.png"))


In [ ]:
# ---- Accuracy table: quad hit rate by horizon vs benchmarks ----------------
import pandas as pd

summary = pd.read_csv("results/summary.csv")
hit = (summary.pivot_table(index="horizon", columns=["basis", "strategy"],
                           values="hit_rate")
       .round(3))
print("Quad hit rate (share of quarters called correctly):")
display(hit)


In [ ]:
# ---- Optional diagnostics (how good is the model, and where does it miss) --
# hit_rate_*: accuracy by horizon vs the persistence / base-effects / random
#             benchmarks. confusion_h*: which quads get confused with which.
# timeline: every call vs what happened, over the whole backtest.
SHOW_DIAGNOSTICS = False

if SHOW_DIAGNOSTICS:
    for p in ["hit_rate_final.png", "hit_rate_first_release.png",
              *sorted(str(x) for x in Path("results").glob("confusion_h*.png")),
              "timeline.png"]:
        p = Path("results") / p if not str(p).startswith("results") else Path(p)
        if p.exists():
            display(Image(filename=str(p)))


In [ ]:
# ---- Optional: download everything (parquet + csv + plots) as a zip --------
import shutil

shutil.make_archive("backtest_output", "zip", "results")
try:
    from google.colab import files
    files.download("backtest_output.zip")
except ImportError:
    print("not running in Colab - backtest_output.zip is in the working dir")


## Going further

- **True GDP vintages:** download the Norges Bank real-time database
  (norges-bank.no -> Statistics), convert it to the CSV format documented in
  `backtest/fetch_data.py`, upload it as `data/gdp_vintages.csv` (Colab file
  pane -> drag & drop), and re-run from the backtest cell. The harness
  switches to `revision_mode='realtime'` automatically.
- **Better market history:** `data/market_monthly.csv` ships with proxies
  (the power column is rescaled from the CPI electricity sub-index).
  Replace it with real Nord Pool / Brent / USDNOK monthly averages and
  re-run - same file, same columns.
- **Every individual prediction** is in `results/backtest_results.parquet`
  (as-of date, target quarter, horizon, quad, deltas, conviction flag,
  benchmark calls, revision-mode flag).
